# Augmentation Plan

1. Collect as many paragraphs as possible from the anonymized ODT sources that contain the labels with the lowest representation in the current training dataset.

2. For each selected paragraph, the data-augmentation pipeline will be:
   - Extract the list of labels present in that paragraph.
   - Keep the distinct labels only, and generate at least 5 Faker candidate values for each distinct label.
   - Map that per-label candidate pool back to the tag occurrences in the paragraph.
   - Ask an LLM to choose which mapped candidate to use for each anonymization tag occurrence, keeping the paragraph coherent.
   - Build the resolved paragraph by replacing the tags with the chosen fake values.
   - Align the anonymized paragraph with the resolved one using `from aymurai.utils.alignment.core import align_text`.
   - Convert the aligned result into BIO lines.

3. A future extension is to augment not only the tags but also the surrounding text itself by generating paraphrases. After collecting enough of those cases, we can run the `ner-langextract-alignment` pipeline again to identify new candidates for the training set.

## What This Notebook Implements

This notebook focuses on the first reliable version of the pipeline:

- Load both anonymized ODT files through the document-extract API.
- Recover the document as normalized paragraphs.
- Parse the ordered less-frequent labels file correctly.
- Select either the first `N` labels from that list or all of them when `TARGET_LABEL_COUNT = None`.
- Keep only paragraphs that contain at least one of those target labels inside `<...>`.
- Extract the labels present in each candidate paragraph.
- Generate 5+ Faker candidates for each distinct label in that paragraph.
- Map the per-label candidates back to the paragraph tag occurrences.
- Ask Ollama to choose replacements per tag occurrence.
- Reconstruct the resolved paragraph locally so alignment stays deterministic.
- Export both structured JSONL samples and a BIO `.txt` file.


In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import uuid
from datetime import datetime
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import requests
from pydantic import BaseModel
from tqdm.auto import tqdm

from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document_api,
)
from aymurai.llm_providers import OllamaLLMProvider
from aymurai.utils.alignment.core import align_text


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "aymurai").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT_DIR = PROJECT_ROOT / "resources" / "outputs" / "data-augmentation"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = OUTPUT_ROOT_DIR / RUN_TIMESTAMP
ALIGNMENTS_DIR = OUTPUT_DIR / "alignments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALIGNMENTS_DIR.mkdir(parents=True, exist_ok=True)

ODT_PATHS = [
    PROJECT_ROOT / "resources" / "data" / "restricted" / "data-augmentation" / "output" / "Resoluciones 2024.odt",
    PROJECT_ROOT / "resources" / "data" / "restricted" / "data-augmentation" / "output" / "Resoluciones 2025.odt",
]

LESS_FREQUENT_LABELS_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "data-preparation"
    / "new-datasets"
    / "top-15-less-frequent-labels.txt"
)
ORIGINAL_TRAIN_UNIQUE_LABELS_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "data-preparation"
    / "new-datasets"
    / "original-train-unique-labels.txt"
)

API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
DOCUMENT_EXTRACT_ENDPOINT = f"{API_BASE_URL}/misc/document-extract"
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:8b")
OLLAMA_KEEP_ALIVE = os.getenv("OLLAMA_KEEP_ALIVE", "5m")
OLLAMA_NUM_CTX = int(os.getenv("OLLAMA_NUM_CTX", "8192"))
OLLAMA_TEMPERATURE = float(os.getenv("OLLAMA_TEMPERATURE", "0"))
ALLOW_OLLAMA_FOR_MISSING_LABELS = os.getenv("ALLOW_OLLAMA_FOR_MISSING_LABELS", "true").lower() == "true"
ALLOW_OLLAMA_NON_FAKER_VALUES = os.getenv("ALLOW_OLLAMA_NON_FAKER_VALUES", "true").lower() == "true"
TARGET_LABEL_COUNT: int | None = None
STRICT_REQUIRE_ALL_RAW_LABELS_MAPPED = True
REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "300"))

JSONL_OUTPUT_PATH = OUTPUT_DIR / "augmented_paragraphs.jsonl"
BIO_OUTPUT_PATH = OUTPUT_DIR / "augmented_train.txt"

PROJECT_ROOT, OUTPUT_DIR


## Step 1: Inventory and standardize labels before selecting candidate paragraphs

Before filtering by the selected less-frequent labels, we first inspect the unique labels that appear across all extracted paragraphs. Some anonymization tags are not standardized yet, so this step builds a raw inventory, applies a normalization map, rewrites the paragraph tags, and only then matches against the chosen target-label slice.

In [ ]:
LABEL_PATTERN = re.compile(r"<([^<>\s]+)>")

LABEL_NORMALIZATION_MAP: dict[str, str] = {
    "CUIJ": "CUIJ",
    "CUIT": "CUIT_CUIL",
    "CUIL": "CUIT_CUIL",
    "CVU": "CBU",
    "EMAIL": "CORREO_ELECTRONICO",
    "MAIL": "CORREO_ELECTRONICO",
    "FECHA_HECHO": "FECHA",
    "NUM": "TELEFONO",
    "NUM_TEL": "TELEFONO",
    "CAUSA": "CUIJ",
    "NUM_CUIT": "CUIT_CUIL",
    "NUMERO_TELEFONO": "TELEFONO",
    "PERIODO": "FECHA",
    "PERÍODO": "FECHA",
    "ACSUADO/A": "PER",
    "ACSUSDO/A": "PER",
    "EMPRESA": "TEXTO_ANONIMIZAR",
    "FECHA_DEL_HECHO": "FECHA",
    "CTA": "NUM_CAJA_AHORRO",
    "NUM_CAUSA": "CUIJ",
    "NUM:CAUSA": "CUIJ",
    "NUM_IPP": "IP",
    "NUM_IP": "IP",
    "INTITUCION": "LOC",
    "INSTITUCIÓN": "LOC",
    "IMEI": "TEXTO_ANONIMIZAR",
    "ALIAS": "TEXTO_ANONIMIZAR",
    "DIR": "DIRECCION",
    "DOMINIO": "PATENTE_DOMINIO",
    "DOMINIO_PATENTE": "PATENTE_DOMINIO",
    "FECHA_NUMERICA": "FECHA",
    "FECHA_NUMÉRICA": "FECHA",
    "NOM": "PER",
    "LUGAR_DE_DETENCIÓN": "LOC",
    "LUGAR_DE_DETENCION": "LOC",
    "PASPORTE": "DNI",
    "PASAPORTE": "DNI",
}

LABEL_RULES: list[tuple[re.Pattern[str], str]] = [
    (re.compile(r"^TEL(?:EFONO)?(?:_|$)|NUM_TEL|NUMERO_TELEFONO|^NUM$"), "TELEFONO"),
    (re.compile(r"EDAD"), "EDAD"),
    (re.compile(r"A+C?S?U?S?AD(?:O|A|X|O_A|A_O)?"), "PER"),
    (re.compile(r"DENUNCIANTE"), "PER"),
    (re.compile(r"DEFENSOR(?:A|X)?"), "PER"),
    (re.compile(r"TESTIG[OA]"), "PER"),
    (re.compile(r"^NOM(?:_|$)"), "PER"),
    (re.compile(r"CLUB"), "LOC"),
    (re.compile(r"INSTITUCION|INTITUCION"), "LOC"),
    (re.compile(r"LOCALIDAD"), "LOC"),
    (re.compile(r"LUGAR_DE_DETENCION"), "LOC"),
    (re.compile(r"LUGAR_HECHO"), "LOC"),
    (re.compile(r"OCUPACION"), "ESTUDIOS"),
    (re.compile(r"PROFESION"), "ESTUDIOS"),
    (re.compile(r"FECHA_NUMERICA|FECHA_HECHO|FECHA_DEL_HECHO|PERIODO"), "FECHA"),
    (re.compile(r"NUM_CAUSA|^CAUSA$"), "CUIJ"),
    (re.compile(r"NUM_CUIT"), "CUIT_CUIL"),
    (re.compile(r"NUM_IPP|NUM_IP"), "IP"),
    (re.compile(r"^CTA(?:_|$)"), "NUM_CAJA_AHORRO"),
    (re.compile(r"^DIR(?:_|$)"), "DIRECCION"),
    (re.compile(r"DOMINIO(?:_PATENTE)?"), "PATENTE_DOMINIO"),
    (re.compile(r"MAIL"), "CORREO_ELECTRONICO"),
    (re.compile(r"ALIAS|IMEI|EMPRESA"), "TEXTO_ANONIMIZAR"),
    (re.compile(r"PASPORTE|PASAPORTE"), "DNI"),
]

@dataclass(frozen=True)
class TagOccurrence:
    occurrence_id: int
    label: str
    start: int
    end: int
    placeholder: str


def load_less_frequent_labels(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df.columns = [str(col).strip() for col in df.columns]
    if "label" not in df.columns:
        raise ValueError(f"Expected a 'label' column in {path}, found columns={df.columns.tolist()}")

    df["label"] = df["label"].astype(str).str.strip()
    if "relative_frequency_pct" in df.columns:
        df["relative_frequency_pct"] = pd.to_numeric(df["relative_frequency_pct"], errors="coerce")

    return df[df["label"].astype(bool)].reset_index(drop=True)


def select_target_label_rows(
    less_frequent_labels_df: pd.DataFrame,
    *,
    target_label_count: int | None,
) -> pd.DataFrame:
    if target_label_count is None:
        return less_frequent_labels_df.copy().reset_index(drop=True)
    if target_label_count <= 0:
        raise ValueError("TARGET_LABEL_COUNT must be positive or None.")
    return less_frequent_labels_df.head(target_label_count).copy().reset_index(drop=True)


def load_unique_labels(path: Path) -> pd.DataFrame:
    labels = [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not labels:
        raise ValueError(f"No labels found in {path}")
    return pd.DataFrame({"label": labels})


def normalize_label_name(label: str, label_map: dict[str, str] | None = None) -> str:
    label = str(label).strip()
    label_map = label_map or {}
    if label in label_map:
        return label_map[label]
    canonical_label = canonicalize_label_for_matching(label)
    if canonical_label.startswith("NUM_"):
        suffix_label = canonical_label[4:]
        if suffix_label in {
            "CAJA_AHORRO",
            "MATRICULA",
            "EXPEDIENTE",
            "ACTUACION",
        }:
            return f"NUM_{suffix_label}"
        if suffix_label in {
            "PER",
            "BANCO",
            "FECHA",
            "DIRECCION",
            "LOC",
            "DNI",
            "TELEFONO",
            "CBU",
            "CUIJ",
            "CUIT_CUIL",
            "IP",
            "LINK",
            "USUARIX",
            "NOMBRE_ARCHIVO",
            "ESTUDIOS",
            "NACIONALIDAD",
            "MARCA_AUTOMOVIL",
            "PATENTE_DOMINIO",
            "TEXTO_ANONIMIZAR",
            "EDAD",
        }:
            return suffix_label
    for pattern, target_label in LABEL_RULES:
        if pattern.search(canonical_label):
            return target_label
    return label


def standardize_paragraph_labels(text: str, label_map: dict[str, str] | None = None) -> str:
    label_map = label_map or {}

    def _replace(match: re.Match[str]) -> str:
        original_label = match.group(1)
        normalized_label = normalize_label_name(original_label, label_map)
        return f"<{normalized_label}>"

    return LABEL_PATTERN.sub(_replace, text)


def build_unique_label_inventory(paragraphs_df: pd.DataFrame, label_column: str) -> pd.DataFrame:
    inventory = (
        paragraphs_df[label_column]
        .explode()
        .dropna()
        .astype(str)
        .value_counts()
        .rename_axis("label")
        .reset_index(name="paragraph_count")
        .sort_values(["paragraph_count", "label"], ascending=[False, True])
        .reset_index(drop=True)
    )
    return inventory


def canonicalize_label_for_matching(label: str) -> str:
    label = str(label).strip().upper()
    label = label.replace("Á", "A").replace("É", "E").replace("Í", "I").replace("Ó", "O").replace("Ú", "U")
    label = re.sub(r"[^A-Z0-9]+", "_", label)
    label = re.sub(r"_+", "_", label).strip("_")
    return label


def build_fuzzy_label_groups(
    train_labels: Iterable[str],
    odt_labels: Iterable[str],
    *,
    min_similarity: float = 0.72,
) -> tuple[dict[str, list[dict[str, Any]]], dict[str, str], pd.DataFrame]:
    train_labels = sorted({str(label).strip() for label in train_labels if str(label).strip()})
    odt_labels = sorted({str(label).strip() for label in odt_labels if str(label).strip()})
    canonical_train_lookup = {
        canonicalize_label_for_matching(train_label): train_label for train_label in train_labels
    }

    grouped_matches: dict[str, list[dict[str, Any]]] = {label: [] for label in train_labels}
    odt_to_train_map: dict[str, str] = {}
    review_rows: list[dict[str, Any]] = []

    for odt_label in odt_labels:
        odt_key = canonicalize_label_for_matching(odt_label)
        if odt_key in canonical_train_lookup:
            exact_train_label = canonical_train_lookup[odt_key]
            grouped_matches[exact_train_label].append({
                "odt_label": odt_label,
                "score": 1.0,
            })
            odt_to_train_map[odt_label] = exact_train_label
            review_rows.append({
                "odt_label": odt_label,
                "best_train_label": exact_train_label,
                "best_score": 1.0,
                "accepted": True,
                "top_3_candidates": [f"{exact_train_label} (1.000)", "exact_canonical_match"],
            })
            continue

        scored_candidates = []
        for train_label in train_labels:
            train_key = canonicalize_label_for_matching(train_label)
            score = SequenceMatcher(None, odt_key, train_key).ratio()
            scored_candidates.append({
                "odt_label": odt_label,
                "train_label": train_label,
                "score": score,
            })

        scored_candidates = sorted(scored_candidates, key=lambda item: (-item["score"], item["train_label"]))
        best_match = scored_candidates[0]
        best_train_label = best_match["train_label"]
        best_score = best_match["score"]
        accepted = best_score >= min_similarity

        review_rows.append({
            "odt_label": odt_label,
            "best_train_label": best_train_label,
            "best_score": best_score,
            "accepted": accepted,
            "top_3_candidates": [
                f"{item['train_label']} ({item['score']:.3f})" for item in scored_candidates[:3]
            ],
        })

        if accepted:
            grouped_matches[best_train_label].append({
                "odt_label": odt_label,
                "score": best_score,
            })
            odt_to_train_map[odt_label] = best_train_label

    grouped_matches = {
        train_label: sorted(matches, key=lambda item: (-item["score"], item["odt_label"]))
        for train_label, matches in grouped_matches.items()
    }

    review_df = pd.DataFrame(review_rows).sort_values(["accepted", "best_score", "odt_label"], ascending=[False, False, True]).reset_index(drop=True)
    return grouped_matches, odt_to_train_map, review_df


def grouped_matches_to_label_map(grouped_matches: dict[str, list[dict[str, Any]]]) -> dict[str, str]:
    label_map: dict[str, str] = {}
    for train_label, matches in grouped_matches.items():
        for match in matches:
            label_map[match["odt_label"]] = train_label
    return label_map


def build_rule_based_label_map(odt_labels: Iterable[str]) -> tuple[dict[str, str], pd.DataFrame]:
    rule_map: dict[str, str] = {}
    rows: list[dict[str, str]] = []
    for odt_label in sorted({str(label).strip() for label in odt_labels if str(label).strip()}):
        normalized = normalize_label_name(odt_label, LABEL_NORMALIZATION_MAP)
        if normalized != odt_label:
            rule_map[odt_label] = normalized
            rows.append({
                "odt_label": odt_label,
                "normalized_label": normalized,
                "match_type": "manual_or_rule",
            })
    return rule_map, pd.DataFrame(rows)


def sort_label_mapping_by_normalized_label(label_map: dict[str, str]) -> dict[str, str]:
    return dict(sorted(label_map.items(), key=lambda item: (item[1], item[0])))


def group_raw_labels_by_normalized_label(label_map: dict[str, str]) -> dict[str, list[str]]:
    grouped: dict[str, list[str]] = {}
    for raw_label, normalized_label in sorted(label_map.items(), key=lambda item: (item[1], item[0])):
        grouped.setdefault(normalized_label, []).append(raw_label)
    return grouped


def build_label_mapping_coverage(
    raw_label_inventory_df: pd.DataFrame,
    effective_label_normalization_map: dict[str, str],
    known_train_labels: set[str],
) -> pd.DataFrame:
    coverage_df = raw_label_inventory_df.copy()
    coverage_df["mapped_label"] = coverage_df["label"].map(
        lambda value: normalize_label_name(value, effective_label_normalization_map)
    )
    coverage_df["is_mapped"] = coverage_df["mapped_label"].isin(known_train_labels)
    coverage_df["mapping_changed"] = coverage_df["mapped_label"] != coverage_df["label"]
    return coverage_df.sort_values(["is_mapped", "paragraph_count", "label"], ascending=[True, False, True]).reset_index(drop=True)


def find_labels_in_text(text: str, allowed_labels: set[str] | None = None) -> list[str]:
    labels = [match.group(1) for match in LABEL_PATTERN.finditer(text)]
    if allowed_labels is None:
        return sorted(set(labels))
    return sorted({label for label in labels if label in allowed_labels})


def extract_tag_occurrences(text: str) -> list[TagOccurrence]:
    occurrences = []
    for idx, match in enumerate(LABEL_PATTERN.finditer(text)):
        occurrences.append(
            TagOccurrence(
                occurrence_id=idx,
                label=match.group(1),
                start=match.start(),
                end=match.end(),
                placeholder=match.group(0),
            )
        )
    return occurrences


def api_extract_document(document_path: Path, session: requests.Session | None = None) -> dict[str, Any]:
    session = session or requests.Session()
    response = extract_document_api(
        session=session,
        endpoint=DOCUMENT_EXTRACT_ENDPOINT,
        file_path=document_path,
        timeout_s=REQUEST_TIMEOUT_S,
    )

    if response.get("status") != "success":
        raise RuntimeError(f"Failed to extract {document_path}: {response.get('detail')}")

    detail = response.get("detail") or {}
    paragraphs = detail.get("document") or []
    return {
        "document_id": detail.get("document_id"),
        "paragraphs": paragraphs,
    }


def collect_paragraphs(
    document_paths: Iterable[Path],
    label_map: dict[str, str] | None = None,
) -> pd.DataFrame:
    label_map = label_map or {}
    records: list[dict[str, Any]] = []
    with requests.Session() as session:
        for document_path in document_paths:
            extracted = api_extract_document(document_path, session=session)
            for paragraph_id, text in enumerate(extracted["paragraphs"]):
                raw_labels = find_labels_in_text(text)
                standardized_text = standardize_paragraph_labels(text, label_map)
                labels = find_labels_in_text(standardized_text)
                records.append(
                    {
                        "source_path": str(document_path),
                        "document_id": extracted["document_id"],
                        "paragraph_id": paragraph_id,
                        "raw_text": text,
                        "text": standardized_text,
                        "raw_labels": raw_labels,
                        "labels": labels,
                    }
                )
    return pd.DataFrame(records)


In [ ]:
FUZZY_LABEL_MATCH_THRESHOLD = 0.72

original_train_unique_labels_df = load_unique_labels(ORIGINAL_TRAIN_UNIQUE_LABELS_PATH)
original_train_unique_labels = set(original_train_unique_labels_df["label"])

less_frequent_labels_df = load_less_frequent_labels(LESS_FREQUENT_LABELS_PATH)
target_labels_df = select_target_label_rows(
    less_frequent_labels_df,
    target_label_count=TARGET_LABEL_COUNT,
)

raw_paragraphs_df = collect_paragraphs(ODT_PATHS, {})
raw_label_inventory_df = build_unique_label_inventory(raw_paragraphs_df, "raw_labels")
rule_based_label_map, rule_based_label_review_df = build_rule_based_label_map(raw_label_inventory_df["label"])
unresolved_odt_labels = [
    label for label in raw_label_inventory_df["label"] if label not in rule_based_label_map
]


fuzzy_label_groups, fuzzy_label_map, fuzzy_label_review_df = build_fuzzy_label_groups(
    train_labels=original_train_unique_labels,
    odt_labels=unresolved_odt_labels,
    min_similarity=FUZZY_LABEL_MATCH_THRESHOLD,
)

effective_label_normalization_map = {
    **fuzzy_label_map,
    **rule_based_label_map,
    **LABEL_NORMALIZATION_MAP,
}
effective_label_normalization_map = sort_label_mapping_by_normalized_label(
    effective_label_normalization_map
)
grouped_raw_labels_by_normalized_label = group_raw_labels_by_normalized_label(
    effective_label_normalization_map
)
(OUTPUT_DIR / "effective_label_normalization_map.json").write_text(
    json.dumps(effective_label_normalization_map, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
(OUTPUT_DIR / "grouped_raw_labels_by_normalized_label.json").write_text(
    json.dumps(grouped_raw_labels_by_normalized_label, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
label_mapping_coverage_df = build_label_mapping_coverage(
    raw_label_inventory_df=raw_label_inventory_df,
    effective_label_normalization_map=effective_label_normalization_map,
    known_train_labels=original_train_unique_labels,
)
unmapped_raw_labels = label_mapping_coverage_df.loc[~label_mapping_coverage_df["is_mapped"], "label"].tolist()
if STRICT_REQUIRE_ALL_RAW_LABELS_MAPPED and unmapped_raw_labels:
    raise ValueError(
        "Some raw ODT labels are still unresolved before replacement: "
        + ", ".join(unmapped_raw_labels)
    )

original_train_unique_labels_df["normalized_label"] = original_train_unique_labels_df["label"].map(
    lambda value: normalize_label_name(value, effective_label_normalization_map)
)
target_labels_df["normalized_label"] = target_labels_df["label"].map(
    lambda value: normalize_label_name(value, effective_label_normalization_map)
)
target_labels = set(target_labels_df["normalized_label"])

paragraphs_df = collect_paragraphs(ODT_PATHS, effective_label_normalization_map)
normalized_label_inventory_df = build_unique_label_inventory(paragraphs_df, "labels")

paragraphs_df["target_labels"] = paragraphs_df["labels"].map(
    lambda labels: [label for label in labels if label in target_labels]
)
paragraphs_df["has_target_label"] = paragraphs_df["target_labels"].map(bool)

candidate_paragraphs_df = (
    paragraphs_df[paragraphs_df["has_target_label"]]
    .copy()
    .sort_values(["source_path", "paragraph_id"])
    .reset_index(drop=True)
)

print(f"Loaded {len(original_train_unique_labels_df)} original train unique labels.")
print(f"Loaded {len(less_frequent_labels_df)} less-frequent labels from file.")
print(f"Using {len(target_labels_df)} target labels for this run.")
print(f"Recovered {len(paragraphs_df)} paragraphs from {len(ODT_PATHS)} ODT files.")
print(f"Found {len(raw_label_inventory_df)} unique raw labels before normalization.")
print(f"Resolved {len(rule_based_label_map)} labels with explicit/rule-based normalization.")
print(f"Accepted {len(fuzzy_label_map)} fuzzy raw-to-train label matches.")
print(f"Mapped {int(label_mapping_coverage_df['is_mapped'].sum())}/{len(label_mapping_coverage_df)} raw labels to known train labels.")
print(f"Found {len(normalized_label_inventory_df)} unique labels after normalization.")
print(f"Selected {len(candidate_paragraphs_df)} paragraphs that contain at least one normalized target label.")


In [ ]:
set(raw_label_inventory_df['label'].to_list())

In [ ]:
(
    original_train_unique_labels_df,
    raw_label_inventory_df.head(50),
    label_mapping_coverage_df,
    rule_based_label_review_df,
    fuzzy_label_review_df.head(50),
    {train_label: matches for train_label, matches in fuzzy_label_groups.items() if matches},
    effective_label_normalization_map,
    grouped_raw_labels_by_normalized_label,
    normalized_label_inventory_df.head(50),
    target_labels_df,
)

In [ ]:
candidate_label_distribution = (
    candidate_paragraphs_df.explode("target_labels")["target_labels"]
    .dropna()
    .value_counts()
    .rename_axis("label")
    .reset_index(name="paragraph_count")
)

candidate_label_distribution

## Step 2: Generate at least 5 Faker options per label

The notebook reuses `aymurai.data_augmentation.anonymizer_entities.augmentation_functions`, which already maps AymurAI labels to Faker-backed generators.


In [ ]:
from aymurai.data_augmentation.anonymizer_entities import (
    augmentation_functions,
    faker as augmentation_faker,
)


LLM_ONLY_LABELS = {"TEXTO_ANONIMIZAR"}


def generate_label_candidates(
    distinct_labels: Iterable[str],
    *,
    n_options: int = 5,
    max_attempts_per_label: int = 50,
    seed: int | None = None,
) -> tuple[dict[str, list[str]], list[str]]:
    if seed is not None:
        augmentation_faker.seed_instance(seed)
    else:
        augmentation_faker.seed_instance(random.randint(1, 1_000_000))

    candidate_map: dict[str, list[str]] = {}
    missing_labels: list[str] = []
    for label in sorted(set(distinct_labels)):
        generator = augmentation_functions.get(label)
        if generator is None:
            missing_labels.append(label)
            continue

        values: list[str] = []
        seen: set[str] = set()
        attempts = 0
        while len(values) < n_options and attempts < max_attempts_per_label:
            attempts += 1
            candidate = str(generator()).strip()
            if not candidate or candidate in seen:
                continue
            values.append(candidate)
            seen.add(candidate)

        if len(values) < n_options:
            raise RuntimeError(
                f"Could not generate {n_options} unique values for label '{label}'. Generated only {len(values)} values."
            )

        candidate_map[label] = values

    unsupported_missing = [label for label in missing_labels if label not in LLM_ONLY_LABELS]
    if unsupported_missing:
        raise ValueError(
            "Unsupported labels without Faker generators: " + ", ".join(sorted(unsupported_missing))
        )

    return candidate_map, sorted(missing_labels)


In [ ]:
sample_labels = sorted({label for labels in candidate_paragraphs_df["labels"] for label in labels})
sample_candidate_map, sample_missing_labels = generate_label_candidates(
    sample_labels,
    n_options=5,
    seed=42,
)
sample_candidate_map, sample_missing_labels


### Missing Label Handling

This notebook assumes that the only label without a Faker generator is `TEXTO_ANONIMIZAR`. That label is intentionally delegated to the LLM. Any other missing label now raises an error immediately so unsupported cases are caught before augmentation.


In [ ]:
LLM_ONLY_LABELS

## Ollama Debug Check

Run this cell before the pipeline if Ollama does not seem to respond. It verifies that the `ollama` Python package is available and that a direct provider call works in the current kernel.


In [ ]:
DEBUG_OLLAMA_MODEL = OLLAMA_MODEL

try:
    import ollama  # noqa: F401
    print("ollama import: ok")
except Exception as exc:
    print(f"ollama import error: {type(exc).__name__}: {exc}")

try:
    debug_provider = OllamaLLMProvider(model=DEBUG_OLLAMA_MODEL, keep_alive=OLLAMA_KEEP_ALIVE)
    debug_response = debug_provider.generate(
        messages=[
            {"role": "system", "content": "Respond with a JSON object only."},
            {"role": "user", "content": '{"status": "ping"}'},
        ],
        options={"temperature": 0, "num_ctx": 2048},
    )
    print("provider call: ok")
    print(debug_response.text)
except Exception as exc:
    print(f"provider call error: {type(exc).__name__}: {exc}")


## Step 3: Ask Ollama to solve the paragraph

In this version, Ollama receives the anonymized paragraph plus the candidate pools, chooses one candidate for each tag occurrence, and returns the final resolved paragraph. The notebook still keeps the chosen replacements so we can align, inspect, and export BIO output.


In [ ]:
def apply_replacements(
    paragraph: str,
    occurrences: list[TagOccurrence],
    replacements: list[dict[str, Any]],
) -> tuple[str, list[dict[str, Any]]]:
    replacement_by_id = {item["occurrence_id"]: item for item in replacements}

    resolved_parts: list[str] = []
    entities: list[dict[str, Any]] = []
    cursor = 0
    resolved_cursor = 0

    for occurrence in occurrences:
        replacement = replacement_by_id[occurrence.occurrence_id]
        chosen_value = replacement["chosen_value"]

        prefix = paragraph[cursor : occurrence.start]
        resolved_parts.append(prefix)
        resolved_cursor += len(prefix)

        entity_start = resolved_cursor
        resolved_parts.append(chosen_value)
        resolved_cursor += len(chosen_value)
        entity_end = resolved_cursor

        entities.append(
            {
                "label": occurrence.label,
                "start_char": entity_start,
                "end_char": entity_end,
                "text": chosen_value,
                "occurrence_id": occurrence.occurrence_id,
                "source_placeholder": occurrence.placeholder,
            }
        )

        cursor = occurrence.end

    suffix = paragraph[cursor:]
    resolved_parts.append(suffix)
    resolved_cursor += len(suffix)

    resolved_text = "".join(resolved_parts)
    return resolved_text, entities


def build_token_offsets(text: str, tokens: list[str]) -> list[tuple[int, int]]:
    offsets: list[tuple[int, int]] = []
    cursor = 0
    for token in tokens:
        start = text.find(token, cursor)
        if start == -1:
            raise ValueError(f"Could not align token '{token}' around '{text[cursor: cursor + 80]}'")
        end = start + len(token)
        offsets.append((start, end))
        cursor = end
    return offsets


def entities_to_bio_lines(text: str, entities: list[dict[str, Any]]) -> list[str]:
    tokens = text.split()
    offsets = build_token_offsets(text, tokens)
    bio_tags = ["O"] * len(tokens)

    for entity in sorted(entities, key=lambda item: (item["start_char"], item["end_char"])):
        first = True
        for idx, (tok_start, tok_end) in enumerate(offsets):
            if entity["end_char"] <= tok_start or entity["start_char"] >= tok_end:
                continue
            prefix = "B-" if first else "I-"
            bio_tags[idx] = f"{prefix}{entity['label']}"
            first = False

    return [f"{token} {label}" for token, label in zip(tokens, bio_tags)]


def alignment_to_bio_lines(mapping: pd.DataFrame) -> list[str]:
    lines: list[str] = []
    for row in mapping.fillna("").to_dict("records"):
        source_chunk = str(row["source"]).strip()
        target_tokens = [token for token in str(row["target"]).split() if token]
        if not target_tokens:
            continue

        match = LABEL_PATTERN.search(source_chunk)
        if not match:
            lines.extend([f"{token} O" for token in target_tokens])
            continue

        label = match.group(1)
        for idx, token in enumerate(target_tokens):
            prefix = "B-" if idx == 0 else "I-"
            lines.append(f"{token} {prefix}{label}")
    return lines


def build_alignment_records(source_text: str, target_text: str) -> tuple[pd.DataFrame, list[dict[str, str]]]:
    alignment_df = align_text(source_text, target_text, columns=("source", "target"))
    alignment_records = alignment_df.fillna("").astype(str).to_dict("records")
    return alignment_df, alignment_records


In [ ]:
class ReplacementSelection(BaseModel):
    occurrence_id: int
    label: str
    chosen_value: str


class ReplacementSelectionBatch(BaseModel):
    resolved_paragraph: str
    replacements: list[ReplacementSelection]


OLLAMA_SYSTEM_PROMPT = """You are helping build a Spanish legal NER training set.
Choose exactly one candidate value for each anonymization tag occurrence.
Resolve the full paragraph by replacing the anonymization tags with the chosen values.
Do not paraphrase the paragraph.
Do not invent new values.
Return only the JSON object required by the schema.
"""

OLLAMA_USER_PROMPT_TEMPLATE = """
Analyze the anonymized paragraph, choose exactly one candidate value for each tag occurrence, and return the final resolved paragraph.

Paragraph:
{paragraph}

Occurrences:
{occurrences_json}

Candidate values by label:
{candidate_values_json}

Missing labels without Faker candidates:
{missing_labels_json}

Rules:
- Use exactly one replacement per occurrence_id.
- If a label exists in candidate_values_by_label, the chosen_value must be copied exactly from that candidate list.
- If a label is listed in missing_labels, you are allowed to generate a coherent replacement directly.
- The field resolved_paragraph must contain the full paragraph with the chosen values inserted in place of the anonymization tags.
- Keep grammatical and legal coherence whenever possible.
- Do not add explanations.
""".strip()


def build_ollama_user_prompt(
    paragraph: str,
    occurrences: list[TagOccurrence],
    candidate_map: dict[str, list[str]],
    missing_labels: list[str],
) -> str:
    occurrences_payload = [
        {
            "occurrence_id": occurrence.occurrence_id,
            "label": occurrence.label,
            "placeholder": occurrence.placeholder,
        }
        for occurrence in occurrences
    ]
    return OLLAMA_USER_PROMPT_TEMPLATE.format(
        paragraph=paragraph,
        occurrences_json=json.dumps(occurrences_payload, ensure_ascii=False, indent=2),
        candidate_values_json=json.dumps(candidate_map, ensure_ascii=False, indent=2),
        missing_labels_json=json.dumps(missing_labels, ensure_ascii=False, indent=2),
    )


def normalize_candidate_text(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[\s_\-\.]+", "", value)
    value = re.sub(r"[^0-9a-záéíóúüñ]", "", value)
    return value


def resolve_candidate_choice(
    label: str,
    chosen_value: str,
    candidate_map: dict[str, list[str]],
    *,
    allow_non_faker_values: bool = False,
) -> tuple[str, bool]:
    candidates = candidate_map.get(label, [])
    if chosen_value in candidates:
        return chosen_value, False

    normalized_choice = normalize_candidate_text(chosen_value)
    normalized_candidates = {}
    for candidate in candidates:
        normalized_candidate = normalize_candidate_text(candidate)
        normalized_candidates.setdefault(normalized_candidate, []).append(candidate)

    matches = normalized_candidates.get(normalized_choice, [])
    if len(matches) == 1:
        return matches[0], False

    if allow_non_faker_values and str(chosen_value).strip():
        return str(chosen_value).strip(), True

    raise ValueError(
        f"Chosen value '{chosen_value}' is not part of the candidates for label '{label}'"
    )


def choose_replacements_with_ollama(
    paragraph: str,
    occurrences: list[TagOccurrence],
    candidate_map: dict[str, list[str]],
    missing_labels: list[str],
    *,
    model: str,
    allow_missing_labels_to_ollama: bool = False,
    allow_non_faker_values: bool = False,
    use_ollama: bool = True,
) -> dict[str, Any]:
    if not use_ollama:
        replacements = [
            {
                "occurrence_id": occurrence.occurrence_id,
                "label": occurrence.label,
                "chosen_value": candidate_map[occurrence.label][0],
                "selection_mode": "fallback_first_candidate",
            }
            for occurrence in occurrences
        ]
        resolved_paragraph, _ = apply_replacements(paragraph, occurrences, replacements)
        return {
            "resolved_paragraph": resolved_paragraph,
            "replacements": replacements,
            "selection_mode": "fallback_first_candidate",
            "user_prompt": None,
            "raw_response_text": None,
        }

    user_prompt = build_ollama_user_prompt(
        paragraph=paragraph,
        occurrences=occurrences,
        candidate_map=candidate_map,
        missing_labels=missing_labels if allow_missing_labels_to_ollama else [],
    )
    provider = OllamaLLMProvider(model=model, keep_alive=OLLAMA_KEEP_ALIVE)
    print("-" * 60)
    print(f"Calling Ollama model: {model}")
    print(f"Occurrences to resolve: {len(occurrences)}")

    response = provider.generate(
        messages=[
            {"role": "system", "content": OLLAMA_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": OLLAMA_TEMPERATURE, "num_ctx": OLLAMA_NUM_CTX},
        format=ReplacementSelectionBatch.model_json_schema(),
    )

    payload = json.loads(response.text)
    replacements = payload.get("replacements")
    resolved_paragraph = str(payload.get("resolved_paragraph", "")).strip()
    if not isinstance(replacements, list):
        raise ValueError(f"Invalid Ollama response payload: {payload}")
    if not resolved_paragraph:
        raise ValueError(f"Missing resolved_paragraph in Ollama response: {payload}")

    validated: list[dict[str, Any]] = []
    expected_ids = {occurrence.occurrence_id for occurrence in occurrences}
    seen_ids: set[int] = set()

    for item in replacements:
        occurrence_id = int(item["occurrence_id"])
        label = str(item["label"])
        chosen_value = str(item["chosen_value"])

        if occurrence_id not in expected_ids:
            raise ValueError(f"Unexpected occurrence_id={occurrence_id}")
        if occurrence_id in seen_ids:
            raise ValueError(f"Duplicate occurrence_id={occurrence_id}")
        if label in candidate_map:
            chosen_value, used_non_faker_value = resolve_candidate_choice(
                label,
                chosen_value,
                candidate_map,
                allow_non_faker_values=allow_non_faker_values,
            )
        elif not (allow_missing_labels_to_ollama and label in missing_labels and chosen_value.strip()):
            raise ValueError(
                f"Label '{label}' has no Faker candidates and Ollama-generated replacements are disabled or empty."
            )
        else:
            used_non_faker_value = True

        seen_ids.add(occurrence_id)
        validated.append(
            {
                "occurrence_id": occurrence_id,
                "label": label,
                "chosen_value": chosen_value,
                "used_non_faker_value": used_non_faker_value,
                "selection_mode": "ollama",
                "model": provider.model_name,
            }
        )

    if seen_ids != expected_ids:
        missing = sorted(expected_ids - seen_ids)
        raise ValueError(f"Missing replacements for occurrence_id values: {missing}")

    return {
        "resolved_paragraph": resolved_paragraph,
        "replacements": sorted(validated, key=lambda item: item["occurrence_id"]),
        "selection_mode": "ollama",
        "contains_non_faker_values": any(item["used_non_faker_value"] for item in validated),
        "user_prompt": user_prompt,
        "raw_response_text": response.text,
    }


## Step 4: Resolve the paragraph, align it, and export BIO lines

The notebook keeps two parallel outputs:

- A structured JSONL file with the augmented paragraph and character-level entity spans.
- A BIO text file built from the same resolved paragraph.

`align_text` is still part of the pipeline because it gives us a transparent source-to-target token mapping for inspection and debugging.


In [ ]:
def augment_paragraph(
    row: pd.Series,
    *,
    candidates_per_label: int = 5,
    use_ollama: bool = True,
) -> dict[str, Any]:
    source_text = str(row["text"])
    occurrences = extract_tag_occurrences(source_text)
    if not occurrences:
        raise ValueError("The paragraph does not contain anonymization tags.")

    labels = [occurrence.label for occurrence in occurrences]
    distinct_labels = sorted(set(labels))
    candidate_map, missing_labels = generate_label_candidates(
        distinct_labels,
        n_options=candidates_per_label,
    )
    if missing_labels and not (use_ollama and ALLOW_OLLAMA_FOR_MISSING_LABELS):
        return {
            "sample_id": uuid.uuid4().hex,
            "document_id": row["document_id"],
            "paragraph_id": int(row["paragraph_id"]),
            "source_path": row["source_path"],
            "source_text": source_text,
            "resolved_text": None,
            "labels": distinct_labels,
            "target_labels": list(row["target_labels"]),
            "candidate_values": candidate_map,
            "missing_labels": missing_labels,
            "status": "skipped_missing_candidate_generators",
            "replacements": [],
            "entities": [],
            "bio_lines": [],
            "alignment_bio_lines": [],
            "alignment_path": None,
            "alignment_records": [],
        }
    llm_resolution = choose_replacements_with_ollama(
        paragraph=source_text,
        occurrences=occurrences,
        candidate_map=candidate_map,
        missing_labels=missing_labels,
        model=OLLAMA_MODEL,
        allow_missing_labels_to_ollama=ALLOW_OLLAMA_FOR_MISSING_LABELS,
        allow_non_faker_values=ALLOW_OLLAMA_NON_FAKER_VALUES,
        use_ollama=use_ollama,
    )

    local_resolved_text, entities = apply_replacements(
        paragraph=source_text,
        occurrences=occurrences,
        replacements=llm_resolution["replacements"],
    )
    resolved_text = llm_resolution["resolved_paragraph"]
    resolved_text_matches_local = resolved_text == local_resolved_text

    alignment_df, alignment_records = build_alignment_records(
        source_text=source_text,
        target_text=resolved_text,
    )
    bio_lines = entities_to_bio_lines(resolved_text, entities)
    alignment_bio_lines = alignment_to_bio_lines(alignment_df)

    sample_id = uuid.uuid4().hex
    ALIGNMENTS_DIR.mkdir(parents=True, exist_ok=True)
    alignment_path = ALIGNMENTS_DIR / f"{sample_id}.csv"
    alignment_df.to_csv(alignment_path, index=False)

    return {
        "sample_id": sample_id,
        "document_id": row["document_id"],
        "paragraph_id": int(row["paragraph_id"]),
        "source_path": row["source_path"],
        "source_text": source_text,
        "resolved_text": resolved_text,
        "labels": distinct_labels,
        "target_labels": list(row["target_labels"]),
        "candidate_values": candidate_map,
        "missing_labels": missing_labels,
        "status": "ok",
        "replacements": llm_resolution["replacements"],
        "llm_selection_mode": llm_resolution["selection_mode"],
        "contains_non_faker_values": llm_resolution.get("contains_non_faker_values", False),
        "ollama_user_prompt": llm_resolution.get("user_prompt"),
        "ollama_raw_response_text": llm_resolution.get("raw_response_text"),
        "local_resolved_text": local_resolved_text,
        "resolved_text_matches_local": resolved_text_matches_local,
        "entities": entities,
        "bio_lines": bio_lines,
        "alignment_bio_lines": alignment_bio_lines,
        "alignment_path": str(alignment_path),
        "alignment_records": alignment_records,
    }


def write_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_bio_txt(path: Path, records: list[dict[str, Any]], *, bio_key: str = "bio_lines") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            if record.get("status") != "ok":
                continue
            for line in record[bio_key]:
                handle.write(line + "\n")
            handle.write("\n")


def run_augmentation_pipeline(
    candidate_rows: pd.DataFrame,
    *,
    max_paragraphs: int | None = None,
    candidates_per_label: int = 5,
    use_ollama: bool = True,
    jsonl_output_path: Path = JSONL_OUTPUT_PATH,
    bio_output_path: Path = BIO_OUTPUT_PATH,
) -> list[dict[str, Any]]:
    if max_paragraphs is not None:
        candidate_rows = candidate_rows.head(max_paragraphs).copy()

    records: list[dict[str, Any]] = []
    progress_total = len(candidate_rows)
    progress = tqdm(total=progress_total, desc="Augmenting paragraphs")

    for _, row in candidate_rows.iterrows():
        record = augment_paragraph(
            row=row,
            candidates_per_label=candidates_per_label,
            use_ollama=use_ollama,
        )
        records.append(record)
        progress.update(1)

    progress.close()

    write_jsonl(jsonl_output_path, records)
    write_bio_txt(bio_output_path, records, bio_key="bio_lines")
    return records


In [ ]:
RUN_PIPELINE = True
USE_OLLAMA = True
ALLOW_OLLAMA_FOR_MISSING_LABELS = True
ALLOW_OLLAMA_NON_FAKER_VALUES = True
MAX_PARAGRAPHS = 2
CANDIDATES_PER_LABEL = 5

if RUN_PIPELINE:
    augmented_records = run_augmentation_pipeline(
        candidate_paragraphs_df[:MAX_PARAGRAPHS],
        max_paragraphs=MAX_PARAGRAPHS,
        candidates_per_label=CANDIDATES_PER_LABEL,
        use_ollama=USE_OLLAMA,
    )
    print(f"Saved {len(augmented_records)} augmented samples to {JSONL_OUTPUT_PATH}")
    print(f"Saved BIO output to {BIO_OUTPUT_PATH}")
else:
    print("Set RUN_PIPELINE = True after the extraction API and Ollama are available.")
    print(f"Planned JSONL output: {JSONL_OUTPUT_PATH}")
    print(f"Planned BIO output: {BIO_OUTPUT_PATH}")


In [ ]:
if JSONL_OUTPUT_PATH.exists():
    preview_records = [json.loads(line) for line in JSONL_OUTPUT_PATH.read_text(encoding="utf-8").splitlines()[:2]]
    preview_records
else:
    print("No JSONL output yet.")

## Pipeline Summary

1. The notebook loads the canonical train-label universe from `original-train-unique-labels.txt` and the ordered less-frequent labels from `top-15-less-frequent-labels.txt`.
2. It extracts every paragraph from the two anonymized ODT files through the document-extract API.
3. It builds the raw ODT label inventory and normalizes those labels with three layers: explicit manual mappings, rule-based mappings, and fuzzy matching only as a fallback for unresolved labels.
4. Before continuing, it verifies that every raw inventory label maps to a known train label. If any raw label is still unresolved, the notebook raises an error and stops.
5. Using the final normalization map, it rewrites the paragraph tags into standardized labels and keeps only the paragraphs that contain at least one of the normalized target labels. `TARGET_LABEL_COUNT` controls how many labels from the less-frequent list are used, and `None` means all of them.
6. For each selected paragraph, it extracts the distinct labels and generates at least 5 Faker candidates for every supported label. The only label intentionally left without Faker generation is `TEXTO_ANONIMIZAR`, which is delegated to the LLM.
7. Ollama receives the anonymized paragraph, the tag occurrences, the Faker candidate pools, and any LLM-only labels. It returns the final resolved paragraph plus one chosen value for each occurrence.
8. If Ollama uses a value that does not exactly match a Faker candidate, the pipeline can still continue when `ALLOW_OLLAMA_NON_FAKER_VALUES` is enabled. Those cases are flagged in each replacement with `used_non_faker_value` and at paragraph level with `contains_non_faker_values`.
9. The notebook aligns the anonymized source paragraph with the resolved paragraph, saves the alignment CSV in the timestamped run folder under `resources/outputs/data-augmentation`, and exports both structured JSONL output and BIO `.txt` output there too.

In short, this pipeline standardizes the anonymization labels first, guarantees coverage of the raw label inventory, resolves one synthetic version per selected paragraph, and exports everything needed to audit and reuse the augmented training data.
